# 04 · Model comparison (both profiles)

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

from boxbox.config import load_config
from boxbox.paths import artifact_paths, dataset_paths
from boxbox.preprocessing.build import read_lap_dataset

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 4)
COLOURS = {"SOFT": "#c93a55", "MEDIUM": "#b07c00", "HARD": "#3b6fb0", "UNKNOWN": "#77766f"}
subset, full = load_config("subset"), load_config("full")

In [ ]:
metrics = json.loads(artifact_paths('subset').combined_metrics.read_text())['profiles']
rows = [dict(profile=p, **m) for p, s in metrics.items() for m in s['metrics'] if m['split'] != 'train']
table = pd.DataFrame(rows).pivot_table(index=['target', 'model'], columns=['profile', 'split'], values='mae')
table.round(3)

Selected model per profile (lowest validation next-lap MAE) and leave-one-race-out CV on the training races:

In [ ]:
pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s['loro_cv_summary']]).pivot(index='model', columns='profile', values='mean').round(3)

In [ ]:
{p: s['selected_model'] for p, s in metrics.items()}

In [ ]:
h = pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s['horizon']])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, profile in zip(axes, ['subset', 'full']):
    for (model, split), g in h[h.profile == profile].groupby(['model', 'split']):
        ax.plot(g.k, g.mae, marker='o', ls='-' if split == 'val' else '--', label=f'{model} {split}')
    ax.set(title=f'{profile}: MAE vs laps ahead', xlabel='k'); ax.legend(fontsize=7)
fig.tight_layout()

## Sequence model (LSTM) vs classical per-lap models

The encoder–decoder LSTM reads the driver's last laps and decodes the known future schedule, so stay-out and pit branches are just different decoder inputs (no frozen-state rollout). It is trained on the training seasons with early stopping on the last two of them; laps beyond its 20-lap decoder horizon follow the tyre prior. Below: horizon MAE, rollout wear slope, training curves, the GRU twin and how it does on pit timing. Whether it wins is reported, not assumed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, profile in zip(axes, ['subset', 'full']):
    g = h[(h.profile == profile) & (h.split == 'val')]
    for model, m in g.groupby('model'):
        ax.plot(m.k, m.mae, marker='o' if model == 'lstm' else '.', lw=2.5 if model == 'lstm' else 1, label=model)
    ax.set(title=f'{profile}: validation MAE vs laps ahead', xlabel='k'); ax.legend(fontsize=7)
fig.tight_layout()

In [ ]:
slopes = pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s.get('rollout_slope', [])])
slopes.pivot_table(index='model', columns=['profile', 'split'], values=['rollout_s_per_lap', 'one_step_s_per_lap']).round(4)

In [ ]:
pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s.get('rollout_slope', []) if r['model'] == 'hgb']).set_index(['profile', 'split'])[['empirical_s_per_lap']].round(4)

Training curves (masked Huber loss; inner validation = last two training seasons, the dashed line marks the kept epoch).

In [ ]:
training = [(p, cell, t) for p, s in metrics.items() for cell, t in s.get('sequence_training', {}).items()]
if training:
    fig, axes = plt.subplots(1, len(training), figsize=(4 * len(training), 3.4), squeeze=False)
    for ax, (profile, cell, t) in zip(axes[0], training):
        curve = pd.DataFrame(t['curve'])
        ax.plot(curve.epoch, curve.train_loss, label='train')
        ax.plot(curve.epoch, curve.inner_val_loss, label=f"inner val {t['inner_val_seasons']}")
        ax.axvline(t['best_epoch'], color='grey', ls='--')
        ax.set(title=f"{profile} {cell}: {t['train_seconds']:.0f} s", xlabel='epoch'); ax.legend(fontsize=7)
    fig.tight_layout()

LSTM vs GRU cell (same hyper-parameters; the GRU is evaluation-only):

In [ ]:
pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s.get('sequence_cell_comparison', [])]).set_index(['profile', 'cell', 'split']).round(3)

Pit-timing check per model (stops at stint lap 5, SC/short/retirement stops excluded; each model with its own validation-calibrated margin):

In [ ]:
pd.DataFrame([
    {'profile': p, 'model': r['model'], 'margin_sigma': r['calibrated_margin_sigma'], 'split': split,
     'in_window_kept': v['in_window_rate_kept'], 'median_width': v['median_width_kept'],
     'stint1_optimum_minus_actual': v['optimum_bias_kept_by_stint'].get('1')}
    for p, s in metrics.items() for r in s.get('pit_timing_by_model', []) for split, v in r['summary'].items()
]).round(3)

## 80% interval coverage

HGB uses quantile gradient boosting (q = 0.1 / 0.9, centred on its point prediction); the other models use ±1.28σ of validation rollout errors, so their validation coverage is in-sample and test is the fair comparison. Target band 0.75–0.85.

In [ ]:
cov = pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s.get('interval_coverage', [])])
cov.pivot_table(index=['target', 'model', 'method'], columns=['profile', 'split'], values=['coverage', 'mean_width_s']).round(3) if len(cov) else cov

Interpretable parameters: polynomial ridge coefficients (seconds) and gradient-boosting permutation importances (validation MAE increase).

In [ ]:
poly = json.loads((artifact_paths('subset').models / 'poly_ridge.meta.json').read_text())['coefficients']
pd.Series(poly).filter(like='tyre_age').round(4)

In [ ]:
hgb = json.loads((artifact_paths('full').models / 'hgb.meta.json').read_text())['importances']
pd.Series(hgb).sort_values(ascending=False).round(4)

In [ ]:
pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s['ablation_deg_features'] + s['fuel_sensitivity']])

In [ ]:
from IPython.display import Image
Image(str(artifact_paths('full').figures / 'pred_vs_actual.png'))